# Synthetic experiments: data-driven robust control for nonlinear systems

Этот ноутбук последовательно выполняет единый pipeline для 4 систем:
1. Генерация данных
2. Идентификация модели `f_hat(x)=Theta(x)C`
3. Анализ residuals и оценка `epsilon`
4. Якобиан и анализ Ляпунова
5. Моделирование с неопределенностью
6. Добавление feedback control и сравнение с uncontrolled


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.control import lqr_gain
from src.identification import fit_least_squares, residual_metrics, residuals
from src.lyapunov import is_hurwitz, jacobian_at_origin, solve_lyapunov_matrix
from src.plots import (
    plot_lyapunov_contours,
    plot_phase_portrait,
    plot_residual_hist,
    plot_trajectories,
)
from src.simulation import generate_dataset, simulate_identified
from src.systems import get_systems
from src.uncertainty import bounding_box
from src.utils import ensure_dirs, save_dataset_csv, save_json

np.random.seed(42)

ensure_dirs('data/synthetic', 'results/figures', 'results/tables', 'results/metrics', 'results/summary')


In [ ]:
T = 20.0
dt = 0.02
t_eval = np.arange(0.0, T + dt, dt)
initial_conditions = [
    np.array([-1.5, -0.5]),
    np.array([-1.0, 1.0]),
    np.array([0.8, -1.2]),
    np.array([1.5, 0.5]),
    np.array([0.5, 1.5]),
]

summary = {'systems': {}}
metrics_rows = []


In [ ]:
for system in get_systems():
    name = system.name
    print(f'\n=== {name} ===')

    data = generate_dataset(system.f, initial_conditions, t_eval)
    x = data['x']
    x_dot = data['x_dot']

    dataset_path = Path(f'data/synthetic/{name}_dataset.csv')
    save_dataset_csv(x, x_dot, data['traj_id'], dataset_path)

    model = fit_least_squares(x, x_dot)
    r = residuals(model, x, x_dot)
    m = residual_metrics(r, quantile=0.95)
    eps = m['epsilon']
    box = bounding_box(x)

    A = jacobian_at_origin(model)
    stable, eigvals = is_hurwitz(A)

    try:
        P = solve_lyapunov_matrix(A)
        lyap_ok = True
    except Exception:
        P = np.eye(2) * np.nan
        lyap_ok = False

    K = lqr_gain(A)

    id_trajectories = []
    for i, x0 in enumerate(initial_conditions):
        t_id, x_id = simulate_identified(model, x0, t_eval, epsilon=0.0, k=None)
        id_trajectories.append({'traj_id': i, 't': t_id, 'x': x_id})

    x0_cmp = np.array([1.2, -1.0])
    t_unc, x_unc = simulate_identified(model, x0_cmp, t_eval, epsilon=eps, k=None)
    t_con, x_con = simulate_identified(model, x0_cmp, t_eval, epsilon=eps, k=K)

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    plot_phase_portrait(ax, data['trajectories'], f'{name}: phase portrait (true)')
    fig.tight_layout()
    fig.savefig(f'results/figures/{name}_phase_true.png', dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    plot_phase_portrait(ax, id_trajectories, f'{name}: phase portrait (identified)')
    fig.tight_layout()
    fig.savefig(f'results/figures/{name}_phase_identified.png', dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    plot_residual_hist(ax, m['residual_norms'], eps, f'{name}: residual histogram')
    fig.tight_layout()
    fig.savefig(f'results/figures/{name}_residual_hist.png', dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    if lyap_ok:
        plot_lyapunov_contours(ax, P, box, f'{name}: Lyapunov contours')
    else:
        ax.text(0.5, 0.5, 'Lyapunov equation failed', ha='center', va='center')
        ax.set_title(f'{name}: Lyapunov contours')
    fig.tight_layout()
    fig.savefig(f'results/figures/{name}_lyapunov_contours.png', dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    plot_trajectories(ax, t_unc, x_unc, f'{name}: uncontrolled trajectories')
    fig.tight_layout()
    fig.savefig(f'results/figures/{name}_uncontrolled.png', dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    plot_trajectories(ax, t_con, x_con, f'{name}: controlled trajectories')
    fig.tight_layout()
    fig.savefig(f'results/figures/{name}_controlled.png', dpi=160)
    plt.close(fig)

    metrics = {
        'system': name,
        'parameters': system.params,
        'rmse': m['rmse'],
        'epsilon': eps,
        'jacobian': A,
        'eigenvalues': eigvals,
        'stable': stable,
        'lyapunov_solved': lyap_ok,
        'P': P,
        'bbox': box,
        'control_results': {
            'K': K,
            'uncontrolled_final_norm': float(np.linalg.norm(x_unc[-1])),
            'controlled_final_norm': float(np.linalg.norm(x_con[-1])),
        },
    }

    save_json(metrics, f'results/metrics/{name}_metrics.json')

    summary['systems'][name] = metrics
    metrics_rows.append({
        'system': name,
        'rmse': m['rmse'],
        'epsilon': eps,
        'stable': stable,
        'lyapunov_solved': lyap_ok,
        'uncontrolled_final_norm': float(np.linalg.norm(x_unc[-1])),
        'controlled_final_norm': float(np.linalg.norm(x_con[-1])),
    })

    print(f"rmse={m['rmse']:.4f}, epsilon={eps:.4f}, stable={stable}")


In [ ]:
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv('results/tables/metrics_overview.csv', index=False)
save_json(summary, 'results/summary/synthetic_experiments_summary.json')
metrics_df


## Артефакты

После выполнения ноутбука сохраняются:
- `data/synthetic/*_dataset.csv`
- `results/figures/*.png`
- `results/metrics/*_metrics.json`
- `results/tables/metrics_overview.csv`
- `results/summary/synthetic_experiments_summary.json`
